In [11]:
import sys
sys.path.append('../utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer
import tiktoken
from collections import defaultdict, Counter
import os
from dotenv import load_dotenv
from nltk.corpus import stopwords
import nltk


In [12]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
print(stop_words)

{'herself', 'himself', "needn't", 'of', 've', "i'd", 'but', 'themselves', 'will', "i've", "she'll", 'your', 'd', "should've", 'they', 'out', 'in', 'hadn', "it's", 'did', 'yours', 'such', 's', 'because', "doesn't", 'a', "they'd", "isn't", 'most', "she'd", 'all', 'been', "don't", 'his', 'for', 'hasn', 'other', 'yourselves', 'o', 'aren', 'mustn', "didn't", 'doesn', "he'd", 'couldn', 'very', "shan't", 'wouldn', 'each', 'or', 'were', 'as', 'at', 'weren', 'him', "that'll", 'what', 'more', 'am', 'on', 'once', 'own', 'from', "i'll", 'then', 'nor', "you'll", 'doing', "mustn't", 'my', 'didn', 'these', 'while', "aren't", 'it', 'ain', 'that', 'until', 'about', "we've", 'm', 'after', 'who', 'is', 'there', 'those', 'under', "you've", 'just', 'by', 'll', 'isn', 'this', "hadn't", "wasn't", 'against', 'itself', 'above', 'so', 'ourselves', 'our', "shouldn't", 'which', 'had', 'before', 'the', 'during', 'how', 'do', 'any', 'here', "weren't", 'through', 'be', 'myself', 'y', 'me', 'haven', 'yourself', 'don'

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pranitgunjal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# **Sentence Transformer**

In [2]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

# **Data Pre-processing**

In [3]:
train_df = pd.read_csv('../data/initial_datasets/reddit/reddit_train.csv')
test_df = pd.read_csv('../data/initial_datasets/reddit/reddit_test.csv')

In [4]:
train_df = train_df.sample(n=1000)

# **Tokenizer**

In [10]:
encoding = tiktoken.encoding_for_model("gpt-4")

# **LLM**

In [5]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

# **Get Co-Occurences**

In [6]:
text = train_df['messages'].to_list()

In [7]:
tokens_list = []
for sentence in text:
    tokens = sentence.split()
    tokens_list.append(tokens)

In [13]:
tokens_list = []
for sentence in text:
    token_ids = encoding.encode(sentence)
    # Optionally, get string versions of tokens
    tokens = [encoding.decode([tid]) for tid in token_ids]
    tokens_list.append(tokens)

In [8]:
cooc = defaultdict(Counter)

for tokens in tokens_list:
    unique_tokens = set(tokens)
    for token in unique_tokens:
        for other_token in unique_tokens:
            if token != other_token:
                cooc[token][other_token] += 1

In [13]:
top_k = 3
summary_text = ""

top_tokens = sorted(cooc.items(), key=lambda item: sum(item[1].values()), reverse=True)[:100]

for token, counter in top_tokens:
    if token in stop_words:
        continue

    top = [w for w, _ in counter.most_common() if w not in stop_words][:top_k]

    if not top:
        continue

    summary_text += f"'{token}' often appears with: {', '.join(top)}.\n"

In [14]:
print(summary_text)

'I' often appears with: like, [NAME], love.
'[NAME]' often appears with: I, [NAME]., like.
'like' often appears with: I, [NAME], looks.
'get' often appears with: I, [NAME], like.
'I'm' often appears with: I, still, going.
'love' often appears with: I, [NAME], much.
'would' often appears with: I, like, [NAME].
'really' often appears with: I, [NAME], like.
'one' often appears with: I, [NAME], guy.
'think' often appears with: I, I'm, would.
'it.' often appears with: I, I'm, But.
'got' often appears with: I, [NAME], like.
'going' often appears with: I, I'm, never.
'This' often appears with: I, could, shit.
'It's' often appears with: I, like, You.
'people' often appears with: I, make, These.
'know' often appears with: I, You, I'm.
'even' often appears with: I, think, [NAME].
'feel' often appears with: I, like, bad.
'You' often appears with: I, know, It's.
'someone' often appears with: I, never, think.
'still' often appears with: I, I'm, [NAME].
'never' often appears with: I, thought, [NAME]

In [15]:
instruction = (
    "You are a data generator tasked with creating realistic Reddit comments. "
    "These comments should be labeled according to their sentiment: positive or negative.\n"
    "Base the style on typical Reddit comments — include informal internet language, typos, abbreviations, and emojis.\n"
    "You will be given information about token co-occurrences which provides information on which words appear near each other.\n"
    "Use [NAME] as a placeholder anytime a person's name would appear.\n"
    "Generate exactly 10 realistic Reddit comments, one per line.\n"
    "Each line should follow this format: the comment in double quotes, followed by a space and then the label (0 for positive, 1 for negative).\n"
    "No extra formatting — just plain text output, one line per comment.\n"
    "Here is the format:\n"
    "\"I love pizza\" 0\n"
    "\"I hate baseball\" 1"
)
input = (
    f"Here are the token co-occurences ordered by frequency:\n{summary_text}",
    f"Now, generate the 10 new comments below:"
)

In [21]:
encoding = tiktoken.encoding_for_model("gpt-4")
print(len(encoding.encode(instruction)))

161


In [24]:
print(len(encoding.encode(input[0])))

57224


In [36]:
response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
print(response.output_text)

1. "Well, I’m glad you enjoyed the game!" 1
2. "I’m so tired of this ridiculous weather." -1
3. "Fantastic job on the project, keep it up!" 1
4. "I can’t believe how bad the movie was..." -1
5. "Honestly, that was the most amazing story!" 1
6. "The customer service here is just awful." -1
7. "I hope your day is going great so far." 1
8. "Why is this game always bugging out? Ugh." -1
9. "You’re doing a fantastic job, congrats!" 1
10. "Ugh, I can’t stand the traffic here." -1


In [16]:
res = []
for i in tqdm(range(200)):
    response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
    res.append(response.output_text)

100%|██████████| 200/200 [17:11<00:00,  5.16s/it]


In [41]:
res

['"I’m so glad I found this community! Everyone here is super helpful and welcoming. 😊" 1  \n"Ugh, this really sucks. I can\'t believe people actually enjoy this nonsense." -1  \n"Wow, I just finished that book and it was absolutely fantastic! Couldn\'t put it down!" 1  \n"I thought my day couldn\'t get any worse, then I read this mess. 😒" -1  \n"I can\'t express how happy I am. Everything went so smoothly today!" 1  \n"This company’s management is so frustrating, nothing ever gets done right. 😠" -1  \n"Glad I joined this project, the team is amazing and I’ve learned a ton!" 1  \n"Honestly, this has been a complete waste of time. Can\'t recommend it at all." -1  \n"Had a great time at the event yesterday. Met some awesome people and learned lots!" 1  \n"I hate when things are so disorganized. It’s just exhausting dealing with it." -1  ',
 '"I\'m so glad you shared that experience! 😊" 1  \n"Honestly, it was a terrible movie." -1  \n"Your post made my day! Thanks for sharing!" 1  \n"Ugh,

In [17]:
labels = []
sentences = []
for i in range(200):
    for word in res[i].split("\n"):
        match = re.match(r'"(.*?)"\s*(-?\d+)', word)
        if match:
            quoted = match.group(1)      
            label = match.group(2)       
            sentences.append(quoted)
            labels.append(int(label))

In [18]:
generated_df = pd.DataFrame({
    'messages': sentences,
    'labels': labels
})

In [19]:
generated_df

,messages,labels
0,I really like the way [NAME] thinks!,0
1,"You know, I'm just not feeling it.",1
2,"Got to say, I love this so much! 😊",0
3,"[NAME], I think you're actually pretty cool!",0
4,"I'm going to have a great day, I can feel it.",0
...,...,...
1995,Would I actually get the chance to see it?,0
1996,People make the funniest posts sometimes 😂,0
1997,This is the worst. I think I'm done.,1
1998,Love how [NAME] always brings good vibes!,0


In [48]:
first_df

,sentences,labels
0,I’m so glad I found this community! Everyone h...,1
1,"Ugh, this really sucks. I can't believe people...",-1
2,"Wow, I just finished that book and it was abso...",1
3,"I thought my day couldn't get any worse, then ...",-1
4,I can't express how happy I am. Everything wen...,1
...,...,...
855,"Well, that’s just a nightmare scenario, sorry.",-1
856,Happy to see improvement in their game.,1
857,I don’t know how to deal with all this stress.,-1
858,"It’s great seeing you here, welcome back!",1


In [50]:
combined = pd.concat([first_df, generated_df])

In [20]:
generated_df.to_csv('../data/generated/reddit/token_co_occurences/new_token_co_occurences.csv', index=False)